# Lesson 1b: From Linear Models to Neurons — Practical

The companion theory notebook (1a) derived logistic regression as a single
neuron from first principles: a weighted sum, a sigmoid, a binary
cross-entropy loss, and a gradient that collapses to the strikingly simple
error signal $(\hat y - y)$. It trained that neuron entirely in NumPy on a
synthetic two-blob dataset, and verified its analytic gradient against a
numerical one.

This notebook reproduces the same model — same computation, same loss, same
gradient — with the production framework (PyTorch), and points it at a real
dataset: MNIST digits 0 and 1, exactly the "0 vs. 1" pixel classification
task 1a's synthetic blobs were designed to mimic.

By the end of this notebook you will have:
- reimplemented 1a's from-scratch NumPy logistic-regression neuron on real
  MNIST 0-vs-1 pixels, as a baseline,
- rebuilt the identical model in PyTorch (`nn.Linear` + `BCEWithLogitsLoss`)
  and confirmed it reaches matching test accuracy,
- wrapped the data in a `Dataset`/`DataLoader` for proper mini-batching, and
- run the **overfit-one-batch** sanity check — the standard first diagnostic
  every practitioner runs before trusting a new training pipeline.

## Introduction

Lesson 1a's argument was mathematical: logistic regression *is* a single
neuron, and its gradient descent update can be derived by hand. That
notebook proved the derivation was correct by checking it against a
numerical finite-difference gradient — but it trained on a synthetic 2D
dataset chosen so the decision boundary could be plotted directly.

Real neurons are trained on real pixels. MNIST's "0 vs. 1" subset is the
smallest possible real version of exactly the same binary classification
problem: 28x28 grayscale images, flattened to a 784-dimensional input
vector, with only two of the ten digit classes kept so the problem stays
binary. Every equation from 1a — $z = w^\top x + b$, $\hat y = \sigma(z)$,
binary cross-entropy, $\nabla_w \mathcal{L} = \frac{1}{n}\sum_i(\hat y_i -
y_i)x_i$ — applies completely unchanged; only the input dimension grows
from 2 to 784.

We first rebuild that exact NumPy model on the real pixels (a direct
continuation of 1a, now on real data) to get a baseline accuracy, then
rebuild the identical model in PyTorch and confirm the two agree. This is
the same "hand-derived vs. framework" comparison lesson 0b ran for a small
MLP, now run for the single-neuron case that 1a introduced.

## Setup

In [ ]:
# Fixed seeds: every stochastic step in this notebook (weight init, data
# shuffling, minibatch order) is reproducible. Seed both numpy and torch,
# and do it before anything random happens.
import numpy as np
import torch

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

import torch.nn as nn
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (5, 4)
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("numpy:", np.__version__)

### Device check

This notebook is written to run unmodified in Google Colab or locally. If a
GPU is available (as it typically is on a Colab GPU runtime) we use it;
otherwise we fall back to CPU. Every tensor and module below is moved to
`device` explicitly.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("using device:", device)

### Loading MNIST and keeping only digits 0 and 1

`torchvision.datasets.MNIST` downloads the dataset automatically the first
time this cell runs, to a local `data/` folder next to this notebook — this
works identically in Colab and locally, no manual setup required. We filter
down to the two classes that make this a binary problem, then subsample
heavily to keep the whole notebook well under the runtime budget on a CPU:
a couple thousand training images and a matching held-out test set, exactly
in the spirit of lesson 0b's "teaching example, not a leaderboard run".

In [ ]:
transform = transforms.Compose([transforms.ToTensor()])

train_full = torchvision.datasets.MNIST(root="data", train=True, download=True, transform=transform)
test_full = torchvision.datasets.MNIST(root="data", train=False, download=True, transform=transform)

def zero_one_subset(dataset, n_per_class, seed):
    labels = dataset.targets.numpy()
    idx0 = np.flatnonzero(labels == 0)
    idx1 = np.flatnonzero(labels == 1)
    g = np.random.default_rng(seed)
    idx0 = g.permutation(idx0)[:n_per_class]
    idx1 = g.permutation(idx1)[:n_per_class]
    idx = g.permutation(np.concatenate([idx0, idx1]))

    images = torch.stack([dataset[i][0] for i in idx])          # (n, 1, 28, 28)
    X = images.reshape(len(idx), -1).numpy().astype(np.float64)  # flatten to 784-d
    y = labels[idx].astype(np.float64)
    return X, y, idx

N_TRAIN_PER_CLASS, N_TEST_PER_CLASS = 1000, 250
X_train, y_train, _ = zero_one_subset(train_full, N_TRAIN_PER_CLASS, seed=SEED)
X_test, y_test, test_idx = zero_one_subset(test_full, N_TEST_PER_CLASS, seed=SEED + 1)

print("X_train:", X_train.shape, " y_train:", y_train.shape, " class balance:", y_train.mean())
print("X_test: ", X_test.shape, " y_test: ", y_test.shape, " class balance:", y_test.mean())

In [ ]:
# A look at the data before we train on it.
fig, axes = plt.subplots(1, 6, figsize=(12, 2.2))
for ax, i in zip(axes, test_idx[:6]):
    img, label = test_full[i]
    ax.imshow(img.squeeze(0), cmap="gray")
    ax.set_title(f"label: {label}", fontsize=9)
    ax.axis("off")
plt.suptitle("MNIST 0-vs-1 samples")
plt.show()

## Reproducing the 1a NumPy Baseline on Real Pixels

Before touching PyTorch, we rerun 1a's *exact* from-scratch logistic
regression — the same `sigmoid`, `bce_loss`, `bce_gradient`, and batch
gradient descent loop — on the flattened MNIST pixels. Pixel values are
scaled to $[0, 1]$ (already true of `ToTensor()` output) which keeps the
gradient well-behaved at 784 input dimensions without any other change to
the algorithm.

In [ ]:
def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-z))

def bce_loss(w, b, X, y, eps=1e-12):
    z = X @ w + b
    yhat = sigmoid(z)
    yhat = np.clip(yhat, eps, 1 - eps)
    return -np.mean(y * np.log(yhat) + (1 - y) * np.log(1 - yhat))

def bce_gradient(w, b, X, y):
    n = X.shape[0]
    z = X @ w + b
    yhat = sigmoid(z)
    error = yhat - y
    grad_w = X.T @ error / n
    grad_b = np.mean(error)
    return grad_w, grad_b

def accuracy(w, b, X, y):
    preds = (sigmoid(X @ w + b) >= 0.5).astype(float)
    return (preds == y).mean()

def gd_batch(X, y, lr, steps, seed):
    g = np.random.default_rng(seed)
    w = g.normal(scale=0.01, size=X.shape[1])
    b = 0.0
    losses = []
    for _ in range(steps):
        grad_w, grad_b = bce_gradient(w, b, X, y)
        w -= lr * grad_w
        b -= lr * grad_b
        losses.append(bce_loss(w, b, X, y))
    return w, b, losses

LR_NUMPY, STEPS_NUMPY = 0.5, 150
w_np, b_np, losses_np = gd_batch(X_train, y_train, lr=LR_NUMPY, steps=STEPS_NUMPY, seed=SEED)

train_acc_np = accuracy(w_np, b_np, X_train, y_train)
test_acc_np = accuracy(w_np, b_np, X_test, y_test)
print(f"NumPy baseline — final train loss: {losses_np[-1]:.4f}")
print(f"NumPy baseline — train accuracy: {train_acc_np:.1%}, test accuracy: {test_acc_np:.1%}")

In [ ]:
plt.plot(losses_np)
plt.xlabel("gradient step")
plt.ylabel("BCE loss")
plt.title("NumPy logistic regression: training loss (MNIST 0-vs-1)")
plt.grid(alpha=0.3)
plt.show()

## The Model in PyTorch

The NumPy model above *is* a single neuron: `X @ w + b` followed by a
sigmoid. In PyTorch this is exactly `nn.Linear(784, 1)` — a fully-connected
layer mapping 784 inputs to a single output logit — with no activation
inside the module. We use `nn.BCEWithLogitsLoss` instead of a separate
sigmoid-then-BCE pair: it applies the sigmoid and the cross-entropy in one
numerically-stable operation (avoiding the `log(0)` risk that our clipped
NumPy version worked around by hand), so its combined gradient is the same
$(\hat y - y)$ signal derived in 1a. This is the smallest possible
`nn.Module` and the same neuron, expressed declaratively instead of as raw
matrix algebra.

In [ ]:
class LogisticNeuron(nn.Module):
    def __init__(self, in_features=28 * 28):
        super().__init__()
        self.linear = nn.Linear(in_features, 1)

    def forward(self, x):
        return self.linear(x).squeeze(-1)   # raw logit z = w^T x + b, shape (batch,)


torch.manual_seed(SEED)
model = LogisticNeuron().to(device)
print(model)
for name, p in model.named_parameters():
    print(f"  {name}: {tuple(p.shape)}")

## DataLoaders and Mini-Batching

1a compared batch, mini-batch, and stochastic gradient descent by
hand-slicing NumPy arrays. PyTorch's `Dataset`/`DataLoader` pair does that
slicing, shuffling, and batching for us. We wrap the flattened pixel tensors
in a `TensorDataset` (the simplest `Dataset`: it just indexes matching
tensors together) and hand it to a `DataLoader`, which yields shuffled
mini-batches each epoch — the same mini-batch idea from 1a, now reusable
infrastructure rather than a hand-written loop.

In [ ]:
Xtr_t = torch.tensor(X_train, dtype=torch.float32)
ytr_t = torch.tensor(y_train, dtype=torch.float32)
Xte_t = torch.tensor(X_test, dtype=torch.float32)
yte_t = torch.tensor(y_test, dtype=torch.float32)

train_dataset = TensorDataset(Xtr_t, ytr_t)
test_dataset = TensorDataset(Xte_t, yte_t)

BATCH_SIZE = 32
g = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, generator=g)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

xb, yb = next(iter(train_loader))
print(f"train dataset: {len(train_dataset)} examples, {len(train_loader)} batches/epoch")
print(f"one mini-batch: xb {tuple(xb.shape)}, yb {tuple(yb.shape)}")

In [ ]:
def train_one_epoch(model, loader, optimizer, loss_fn, device):
    model.train()
    total_loss, n_batches = 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1
    return total_loss / n_batches


@torch.no_grad()
def evaluate_accuracy(model, loader, device):
    model.eval()
    correct, total = 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        preds = (torch.sigmoid(model(xb)) >= 0.5).float()
        correct += (preds == yb).sum().item()
        total += yb.size(0)
    return correct / total


loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

N_EPOCHS = 15
train_losses = []
for epoch in range(1, N_EPOCHS + 1):
    epoch_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, device)
    train_losses.append(epoch_loss)

train_acc_torch = evaluate_accuracy(model, train_loader, device)
test_acc_torch = evaluate_accuracy(model, test_loader, device)
print(f"PyTorch neuron — final epoch loss: {train_losses[-1]:.4f}")
print(f"PyTorch neuron — train accuracy: {train_acc_torch:.1%}, test accuracy: {test_acc_torch:.1%}")

In [ ]:
plt.plot(range(1, N_EPOCHS + 1), train_losses, marker="o")
plt.xlabel("epoch")
plt.ylabel("mean training BCE loss")
plt.title("PyTorch logistic neuron: training loss (MNIST 0-vs-1)")
plt.xticks(range(1, N_EPOCHS + 1))
plt.grid(alpha=0.3)
plt.show()

### Comparing the two implementations

Same model, same loss, same optimisation problem, two implementations
14 years apart in tooling. The test accuracies should be close — small
differences are expected from different initialisations, optimiser step
counts (150 full-batch steps vs. 15 mini-batch epochs), and floating point
precision (`float64` in NumPy vs. `float32` in PyTorch), not from any
difference in the underlying model.

In [ ]:
print(f"{'implementation':<12}{'train acc':>12}{'test acc':>12}")
print(f"{'NumPy':<12}{train_acc_np:>12.1%}{test_acc_np:>12.1%}")
print(f"{'PyTorch':<12}{train_acc_torch:>12.1%}{test_acc_torch:>12.1%}")

gap = abs(test_acc_np - test_acc_torch)
print(f"\ntest accuracy gap: {gap:.1%}")
assert test_acc_np > 0.9, "NumPy baseline should comfortably beat chance (50%) on 0-vs-1"
assert test_acc_torch > 0.9, "PyTorch model should comfortably beat chance (50%) on 0-vs-1"
assert gap < 0.05, "the two implementations of the same model should reach closely matching accuracy"
print("Confirmed: the PyTorch reimplementation matches the from-scratch NumPy neuron's accuracy.")

## Overfit One Batch

Before trusting any new training pipeline, practitioners run one
standard sanity check: take a **single small batch**, turn off everything
that regularises or generalises, and train on *only that batch* for many
steps. A correctly-wired model with enough capacity for the batch size
should drive its loss to (near) zero and its accuracy on that exact batch
to 100% — because with no other data to reconcile against, there is
nothing stopping it from memorising 16-32 examples outright.

This does **not** test whether the model generalises. It tests whether the
training mechanics — the forward pass, the loss, the backward pass, the
optimiser step — are wired correctly at all. If a model *cannot* drive the
loss on one tiny batch to near zero, the bug is almost certainly structural:
a frozen or disconnected parameter (e.g. gradients never reaching the
weights because of a broken computational graph), a loss function mismatched
to the model's output (e.g. feeding probabilities into a loss that expects
logits, or vice versa), labels and inputs misaligned by a shuffling bug, or
a learning rate so small the optimiser cannot make visible progress in any
reasonable number of steps. Overfitting one batch successfully is a cheap,
fast (seconds, not minutes) precondition that should always be checked
*before* spending a full training run's time debugging why a model "isn't
learning" on the whole dataset.

In [ ]:
# A single small batch, held fixed for every step below.
torch.manual_seed(SEED)
OVERFIT_BATCH_SIZE = 24
overfit_loader = DataLoader(train_dataset, batch_size=OVERFIT_BATCH_SIZE, shuffle=True,
                             generator=torch.Generator().manual_seed(SEED))
xb_fixed, yb_fixed = next(iter(overfit_loader))
xb_fixed, yb_fixed = xb_fixed.to(device), yb_fixed.to(device)
print(f"overfit batch: {xb_fixed.shape[0]} examples, class balance {yb_fixed.mean().item():.2f}")

overfit_model = LogisticNeuron().to(device)
overfit_optimizer = torch.optim.SGD(overfit_model.parameters(), lr=0.5)

OVERFIT_STEPS = 300
overfit_losses, overfit_accs = [], []
for step in range(OVERFIT_STEPS):
    overfit_model.train()
    overfit_optimizer.zero_grad()
    logits = overfit_model(xb_fixed)
    loss = loss_fn(logits, yb_fixed)
    loss.backward()
    overfit_optimizer.step()

    with torch.no_grad():
        preds = (torch.sigmoid(logits) >= 0.5).float()
        acc = (preds == yb_fixed).float().mean().item()
    overfit_losses.append(loss.item())
    overfit_accs.append(acc)

print(f"overfit-one-batch — step 0 loss: {overfit_losses[0]:.4f}, acc: {overfit_accs[0]:.1%}")
print(f"overfit-one-batch — final loss: {overfit_losses[-1]:.6f}, acc: {overfit_accs[-1]:.1%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(overfit_losses)
axes[0].set_xlabel("step"); axes[0].set_ylabel("BCE loss on the fixed batch")
axes[0].set_title("Overfit-one-batch: loss -> 0")
axes[0].grid(alpha=0.3)

axes[1].plot(overfit_accs)
axes[1].set_xlabel("step"); axes[1].set_ylabel("accuracy on the fixed batch")
axes[1].set_title("Overfit-one-batch: accuracy -> 100%")
axes[1].set_ylim(-0.05, 1.05)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

assert overfit_losses[-1] < 1e-2, "loss on the single fixed batch should collapse to near zero"
assert overfit_accs[-1] == 1.0, "accuracy on the single fixed batch should reach 100%"
print("Confirmed: the training pipeline correctly memorises a single batch — "
      "loss -> ~0 and accuracy -> 100% on that batch.")

Had this check *failed* — loss plateauing well above zero, or accuracy
stuck below 100% after hundreds of steps on just 24 examples — that would be
strong evidence of a wiring bug rather than a genuinely hard learning
problem, because 24 points are trivially separable by a 784-parameter linear
model. The usual suspects, in the order a practitioner should check them:
gradients not flowing to some parameter (check `.grad` is non-`None` and
non-zero after `.backward()`), a loss/model output mismatch (logits fed to a
loss expecting probabilities, or the reverse), or inputs and labels silently
misaligned (e.g. a shuffle applied to one tensor but not the other, which the
`TensorDataset` above rules out by construction since it indexes both
tensors together).

## Key Takeaways

- Logistic regression's from-scratch NumPy form and its PyTorch form
  (`nn.Linear` + `BCEWithLogitsLoss`) are **the same model, the same loss,
  and the same gradient** — we confirmed this by training both on the same
  real data (MNIST 0-vs-1) and finding matching test accuracy, not just
  matching equations on paper.
- `nn.BCEWithLogitsLoss` combines the sigmoid and the cross-entropy into one
  numerically stable operation, so the model itself outputs a raw logit
  rather than a probability — the same $(\hat y - y)$ gradient 1a derived by
  hand is what PyTorch's autograd computes automatically.
- `Dataset` and `DataLoader` replace hand-written batch-slicing code with
  reusable, shuffled mini-batching — the same mini-batch idea from 1a, now
  infrastructure rather than a bespoke loop.
- **Overfit one batch** before trusting a new pipeline on the full dataset:
  a correctly-wired model should drive loss to ~0 and accuracy to 100% on a
  single small, fixed batch in seconds. Failure to do so points at broken
  training mechanics (gradients not flowing, a loss/output mismatch, or
  misaligned data) rather than a genuinely hard learning problem.
- Everything here — the single neuron, its loss, `Dataset`/`DataLoader`
  mini-batching, and the overfit-one-batch check — carries over unchanged
  when we stack many neurons into a multi-layer network and derive
  backpropagation by hand in the next lesson (2a).